# Análise da Camada Gold — Pipeline de Alfabetização

Visualizações dos dados analíticos do pipeline híbrido (Batch + Streaming) para análise da alfabetização no Brasil.

**Fonte:** Indicador Criança Alfabetizada (INEP/SAEB) via Base dos Dados + Microdados AEEB 2025

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configuração visual
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Carregar tabelas Gold
panorama = pd.read_csv('../../tabelas_gold/panorama_alfabetizacao.csv')
evolucao = pd.read_csv('../../tabelas_gold/evolucao_proficiencia.csv')
ranking = pd.read_csv('../../tabelas_gold/ranking_municipios.csv')
indicadores = pd.read_csv('../../tabelas_gold/indicadores_alunos.csv')
indicadores_mun = pd.read_csv('../../tabelas_gold/indicadores_alunos_municipio.csv')
desigualdade = pd.read_csv('../../tabelas_gold/desigualdade_regional.csv')
criticos = pd.read_csv('../../tabelas_gold/municipios_criticos.csv')
tendencia = pd.read_csv('../../tabelas_gold/tendencia_temporal.csv')
features = pd.read_csv('../../tabelas_gold/features_ml.csv')

print('Tabelas carregadas com sucesso!')
print(f'panorama: {len(panorama)} registros')
print(f'desigualdade: {len(desigualdade)} registros')
print(f'criticos: {len(criticos)} registros')
print(f'tendencia: {len(tendencia)} registros')
print(f'indicadores: {len(indicadores)} registros')

: 

## 1. Panorama Nacional — Taxa vs Meta 2030

In [ ]:
# Evolução Brasil
brasil = panorama[(panorama['granularidade'] == 'Brasil')].sort_values('ano')

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(brasil['ano'].astype(str), brasil['taxa_alfabetizacao'], color='#2196F3', width=0.5, label='Taxa Realizada')
ax.axhline(y=80, color='#E53935', linestyle='--', linewidth=2, label='Meta 2030 (80%)')
ax.set_xlabel('Ano')
ax.set_ylabel('Taxa de Alfabetização (%)')
ax.set_title('Brasil — Taxa de Alfabetização vs Meta 2030', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100)
ax.legend()

for i, row in brasil.iterrows():
    if pd.notna(row['taxa_alfabetizacao']):
        ax.text(str(int(row['ano'])), row['taxa_alfabetizacao'] + 2, f"{row['taxa_alfabetizacao']:.1f}%", 
                ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Insights
taxa_atual = brasil['taxa_alfabetizacao'].iloc[-1]
taxa_primeira = brasil['taxa_alfabetizacao'].iloc[0]
crescimento_total = taxa_atual - taxa_primeira
anos_dados = len(brasil)
crescimento_anual_medio = crescimento_total / (anos_dados - 1) if anos_dados > 1 else 0
gap_atual = 80 - taxa_atual
anos_restantes = 2030 - brasil['ano'].iloc[-1]
crescimento_necessario = gap_atual / anos_restantes if anos_restantes > 0 else 0

print('\n' + '='*60)
print('INSIGHTS — PANORAMA NACIONAL')
print('='*60)
print(f'Taxa atual (último ano disponível): {taxa_atual:.1f}%')
print(f'Meta 2030: 80%')
print(f'Gap atual para a meta: {gap_atual:.1f} pontos percentuais')
print(f'Crescimento acumulado no período: +{crescimento_total:.1f} p.p.')
print(f'Crescimento médio anual observado: +{crescimento_anual_medio:.1f} p.p./ano')
print(f'Crescimento necessário por ano até 2030: +{crescimento_necessario:.1f} p.p./ano')
if crescimento_anual_medio >= crescimento_necessario:
    print(f'\n✓ No ritmo atual, o Brasil ATINGIRÁ a meta de 2030.')
else:
    print(f'\n⚠ No ritmo atual, o Brasil NÃO atingirá a meta.')
    print(f'  Precisa acelerar {crescimento_necessario - crescimento_anual_medio:.1f} p.p./ano a mais.')
print(f'\nParticipação na avaliação: {brasil["percentual_participacao"].iloc[-1]:.0f}% dos alunos')

## 2. Desigualdade Regional

In [ ]:
# Desigualdade entre regiões (2024)
desig_2024 = desigualdade[desigualdade['ano'] == 2024].sort_values('media_taxa_alfabetizacao', ascending=True)

if len(desig_2024) == 0:
    desig_2024 = desigualdade[desigualdade['ano'] == 2023].sort_values('media_taxa_alfabetizacao', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
cores = {'Norte': '#FF7043', 'Nordeste': '#FFA726', 'Centro-Oeste': '#FFEE58', 'Sudeste': '#66BB6A', 'Sul': '#42A5F5'}
barras = ax.barh(desig_2024['regiao'], desig_2024['media_taxa_alfabetizacao'], 
                 color=[cores.get(r, '#999') for r in desig_2024['regiao']], height=0.6)
ax.axvline(x=80, color='#E53935', linestyle='--', linewidth=2, label='Meta 2030 (80%)')

for bar, gap in zip(barras, desig_2024['gap_medio_meta_2030']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
            f'Gap: {gap:.1f}', va='center', fontsize=10)

ax.set_xlabel('Média da Taxa de Alfabetização (%)')
ax.set_title('Desigualdade Regional — Taxa Média de Alfabetização', fontsize=14, fontweight='bold')
ax.set_xlim(0, 100)
ax.legend()
plt.tight_layout()
plt.show()

# Insights
melhor_reg = desig_2024.iloc[-1]
pior_reg = desig_2024.iloc[0]
diferenca = melhor_reg['media_taxa_alfabetizacao'] - pior_reg['media_taxa_alfabetizacao']
print('\n' + '='*60)
print('INSIGHTS — DESIGUALDADE REGIONAL')
print('='*60)
print(f'Região com melhor desempenho: {melhor_reg["regiao"]} ({melhor_reg["media_taxa_alfabetizacao"]:.1f}%)')
print(f'Região com pior desempenho: {pior_reg["regiao"]} ({pior_reg["media_taxa_alfabetizacao"]:.1f}%)')
print(f'Diferença entre melhor e pior região: {diferenca:.1f} pontos percentuais')
print(f'\nGap médio por região:')
for _, row in desig_2024.sort_values('gap_medio_meta_2030', ascending=False).iterrows():
    print(f'  {row["regiao"]:15s} → Gap: {row["gap_medio_meta_2030"]:.1f} p.p. | Amplitude interna: {row["amplitude"]:.1f}')
print(f'\n⚠ A amplitude interna alta indica que mesmo dentro da mesma região há municípios em situações muito distintas.')
print(f'⚠ Norte e Nordeste precisam de investimento {(pior_reg["gap_medio_meta_2030"]/melhor_reg["gap_medio_meta_2030"]):.1f}x maior que o Sul para atingir a meta.')

## 3. Top UFs — Maior e Menor Taxa (2024)

In [ ]:
# UFs em 2024
ufs_2024 = panorama[(panorama['granularidade'] == 'UF') & (panorama['ano'] == 2024)].copy()
ufs_2024 = ufs_2024.dropna(subset=['taxa_alfabetizacao']).sort_values('taxa_alfabetizacao', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
cores_uf = ['#E53935' if t < 50 else '#FFA726' if t < 65 else '#66BB6A' if t < 80 else '#1B5E20' 
            for t in ufs_2024['taxa_alfabetizacao']]
ax.barh(ufs_2024['codigo'], ufs_2024['taxa_alfabetizacao'], color=cores_uf)
ax.axvline(x=80, color='#E53935', linestyle='--', linewidth=2, label='Meta 2030')
ax.set_xlabel('Taxa de Alfabetização (%)')
ax.set_title('Taxa de Alfabetização por UF — 2024', fontsize=14, fontweight='bold')
ax.set_xlim(0, 100)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Municípios Críticos — Distribuição por Nível de Criticidade

In [ ]:
# Distribuição de criticidade
crit_count = criticos['nivel_criticidade'].value_counts()
ordem = ['Crítico', 'Muito Alto', 'Alto', 'Moderado', 'Baixo']
crit_count = crit_count.reindex([o for o in ordem if o in crit_count.index])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pizza
cores_crit = ['#B71C1C', '#E53935', '#FF7043', '#FFA726', '#66BB6A']
axes[0].pie(crit_count.values, labels=crit_count.index, autopct='%1.1f%%', 
            colors=cores_crit[:len(crit_count)], startangle=90)
axes[0].set_title('Distribuição por Nível de Criticidade', fontweight='bold')

# Por região
crit_regiao = criticos.groupby(['regiao', 'nivel_criticidade']).size().unstack(fill_value=0)
crit_regiao = crit_regiao.reindex(columns=[o for o in ordem if o in crit_regiao.columns])
crit_regiao.plot(kind='bar', stacked=True, ax=axes[1], color=cores_crit[:len(crit_regiao.columns)])
axes[1].set_title('Municípios Críticos por Região', fontweight='bold')
axes[1].set_xlabel('Região')
axes[1].set_ylabel('Quantidade de Municípios')
axes[1].legend(title='Criticidade', bbox_to_anchor=(1.05, 1))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Insights municípios críticos
total_mun = len(criticos)
criticos_nordeste = len(criticos[criticos['regiao'] == 'Nordeste'])
criticos_norte = len(criticos[criticos['regiao'] == 'Norte'])
pct_norte_nordeste = (criticos_nordeste + criticos_norte) / total_mun * 100
media_gap = criticos['gap_meta_2030'].mean()
taxa_media_criticos = criticos['taxa_alfabetizacao'].mean()
mun_abaixo_20 = len(criticos[criticos['taxa_alfabetizacao'] < 20])
mun_abaixo_50 = len(criticos[criticos['taxa_alfabetizacao'] < 50])

print('\n' + '='*60)
print('INSIGHTS — MUNICÍPIOS CRÍTICOS')
print('='*60)
print(f'Total de municípios analisados: {total_mun}')
print(f'Gap médio para meta 2030: {media_gap:.1f} pontos')
print(f'Taxa média de alfabetização: {taxa_media_criticos:.1f}%')
print(f'\nConcentração geográfica:')
print(f'  Nordeste: {criticos_nordeste} municípios ({criticos_nordeste/total_mun*100:.0f}%)')
print(f'  Norte: {criticos_norte} municípios ({criticos_norte/total_mun*100:.0f}%)')
print(f'  Norte + Nordeste: {pct_norte_nordeste:.0f}% dos municípios críticos')
print(f'\nSituação grave:')
print(f'  Municípios com taxa < 20%: {mun_abaixo_20} (precisam 4x mais esforço)')
print(f'  Municípios com taxa < 50%: {mun_abaixo_50} ({mun_abaixo_50/total_mun*100:.0f}% do total)')
print(f'\n⚠ Os municípios mais críticos precisariam crescer {media_gap/(2030-2024):.1f} p.p./ano para atingir a meta — taxa muito acima da média nacional.')

In [ ]:
# Status de atingimento por UF
tend_2024 = tendencia[tendencia['ano'] == 2024].copy()

status_count = tend_2024['vai_atingir_2030'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pizza de status
cores_status = {'Já atingiu': '#1B5E20', 'Sim': '#66BB6A', 'Não no ritmo atual': '#E53935', 'Sem dados': '#BDBDBD'}
axes[0].pie(status_count.values, labels=status_count.index, autopct='%1.0f%%',
            colors=[cores_status.get(s, '#999') for s in status_count.index], startangle=90)
axes[0].set_title('UFs — Projeção de Atingimento da Meta 2030', fontweight='bold')

# Variação anual por UF
tend_var = tend_2024.dropna(subset=['variacao_anual']).sort_values('variacao_anual')
cores_var = ['#E53935' if v < 0 else '#66BB6A' for v in tend_var['variacao_anual']]
axes[1].barh(tend_var['sigla_uf'], tend_var['variacao_anual'], color=cores_var)
axes[1].axvline(x=0, color='black', linewidth=0.5)
axes[1].set_xlabel('Variação Anual (pontos percentuais)')
axes[1].set_title('Variação Anual da Taxa por UF (2023→2024)', fontweight='bold')

plt.tight_layout()
plt.show()

# Insights tendência
ja_atingiu = len(tend_2024[tend_2024['vai_atingir_2030'] == 'Já atingiu'])
vai_atingir = len(tend_2024[tend_2024['vai_atingir_2030'] == 'Sim'])
nao_atinge = len(tend_2024[tend_2024['vai_atingir_2030'] == 'Não no ritmo atual'])
sem_dados = len(tend_2024[tend_2024['vai_atingir_2030'] == 'Sem dados'])
regrediu = tend_2024[tend_2024['variacao_anual'] < 0]
maior_crescimento = tend_2024.dropna(subset=['variacao_anual']).nlargest(3, 'variacao_anual')

print('\n' + '='*60)
print('INSIGHTS — TENDÊNCIA TEMPORAL')
print('='*60)
print(f'Status de atingimento da meta 2030:')
print(f'  Já atingiu: {ja_atingiu} UFs')
print(f'  Vai atingir no ritmo atual: {vai_atingir} UFs')
print(f'  NÃO vai atingir no ritmo atual: {nao_atinge} UFs')
print(f'  Sem dados suficientes: {sem_dados} UFs')
print(f'\nUFs que REGREDIRAM (taxa caiu):')
for _, row in regrediu.iterrows():
    print(f'  {row["sigla_uf"]}: variação de {row["variacao_anual"]:.1f} p.p.')
print(f'\nTop 3 UFs com MAIOR crescimento:')
for _, row in maior_crescimento.iterrows():
    print(f'  {row["sigla_uf"]}: +{row["variacao_anual"]:.1f} p.p. (projeção: {row["vai_atingir_2030"]})')
print(f'\n⚠ Estados com variação negativa precisam de intervenção urgente — estão se afastando da meta.')

In [ ]:
# Taxa de alfabetização por UF (dados streaming)
# Agrupar por UF (somar todas as redes) para visão limpa
ind_agrupado = indicadores.groupby('sg_uf').agg(
    total_alunos=('total_alunos_avaliados', 'sum'),
    total_alfabetizados=('total_alfabetizados', 'sum'),
    media_prof=('media_proficiencia', 'mean')
).reset_index()
ind_agrupado['taxa'] = (ind_agrupado['total_alfabetizados'] / ind_agrupado['total_alunos'] * 100).round(1)
ind_agrupado = ind_agrupado.sort_values('taxa', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Gráfico 1: Taxa por UF (limpo, uma barra por estado)
cores_ind = ['#E53935' if t < 50 else '#FFA726' if t < 65 else '#66BB6A' if t < 80 else '#1B5E20' 
             for t in ind_agrupado['taxa']]
bars = axes[0].barh(ind_agrupado['sg_uf'], ind_agrupado['taxa'], color=cores_ind, height=0.7)
axes[0].axvline(x=80, color='#E53935', linestyle='--', linewidth=2, label='Meta 2030 (80%)')
axes[0].axvline(x=743/10, color='#FF9800', linestyle=':', linewidth=1.5)
axes[0].set_xlabel('Taxa de Alfabetização (%)')
axes[0].set_title('AEEB 2025 — Taxa de Alfabetização por UF\n(Dados Streaming)', fontsize=12, fontweight='bold')
axes[0].set_xlim(0, 100)
axes[0].legend(loc='lower right')
# Adicionar valor na barra
for bar, val in zip(bars, ind_agrupado['taxa']):
    axes[0].text(val + 1, bar.get_y() + bar.get_height()/2, f'{val:.0f}%', va='center', fontsize=8)

# Gráfico 2: Boxplot de proficiência
axes[1].hist(indicadores['media_proficiencia'], bins=12, color='#42A5F5', edgecolor='white', alpha=0.8)
axes[1].axvline(x=743, color='#E53935', linestyle='--', linewidth=2, label='Ponto de corte (743)')
media_geral = indicadores['media_proficiencia'].mean()
axes[1].axvline(x=media_geral, color='#1B5E20', linestyle='-', linewidth=2, label=f'Média geral ({media_geral:.0f})')
axes[1].set_xlabel('Média de Proficiência (escala SAEB)')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição de Proficiência Média\npor UF/Rede', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

# Insights
melhor = ind_agrupado.iloc[-1]
pior = ind_agrupado.iloc[0]
total_alunos = indicadores['total_alunos_avaliados'].sum()
total_alfab = indicadores['total_alfabetizados'].sum()
print(f'\n--- INSIGHTS STREAMING AEEB 2025 ---')
print(f'Total de alunos avaliados: {total_alunos:,.0f}')
print(f'Total alfabetizados: {total_alfab:,.0f} ({total_alfab/total_alunos*100:.1f}%)')
print(f'Melhor UF: {melhor["sg_uf"]} ({melhor["taxa"]:.1f}%)')
print(f'Pior UF: {pior["sg_uf"]} ({pior["taxa"]:.1f}%)')
print(f'Diferença entre melhor e pior: {melhor["taxa"] - pior["taxa"]:.1f} pontos')
print(f'UFs acima da meta (80%): {len(ind_agrupado[ind_agrupado["taxa"] >= 80])}')
print(f'UFs abaixo de 50%: {len(ind_agrupado[ind_agrupado["taxa"] < 50])}')

## 7. Gap da Meta 2030 — Distribuição dos Municípios

In [ ]:
# Histograma do gap
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(criticos['gap_meta_2030'], bins=30, color='#FF7043', edgecolor='white', alpha=0.8)
ax.axvline(x=criticos['gap_meta_2030'].median(), color='#1B5E20', linestyle='--', 
           linewidth=2, label=f"Mediana: {criticos['gap_meta_2030'].median():.1f}")
ax.set_xlabel('Gap para Meta 2030 (pontos percentuais)')
ax.set_ylabel('Quantidade de Municípios')
ax.set_title('Distribuição do Gap para Meta 2030 — Todos os Municípios', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nEstatísticas do Gap:")
print(f"  Média: {criticos['gap_meta_2030'].mean():.1f} pontos")
print(f"  Mediana: {criticos['gap_meta_2030'].median():.1f} pontos")
print(f"  Municípios com gap > 50: {len(criticos[criticos['gap_meta_2030'] > 50])}")
print(f"  Municípios com gap < 10: {len(criticos[criticos['gap_meta_2030'] < 10])}")
print(f"  Desvio padrão: {criticos['gap_meta_2030'].std():.1f}")
print(f"  Gap mínimo: {criticos['gap_meta_2030'].min():.1f} (município mais próximo da meta)")
print(f"  Gap máximo: {criticos['gap_meta_2030'].max():.1f} (município mais distante)")
print(f"\n  75% dos municípios têm gap maior que {criticos['gap_meta_2030'].quantile(0.25):.1f} pontos")
print(f"  50% dos municípios têm gap maior que {criticos['gap_meta_2030'].quantile(0.50):.1f} pontos")
print(f"  25% dos municípios têm gap maior que {criticos['gap_meta_2030'].quantile(0.75):.1f} pontos")
print(f"\n⚠ A distribuição é assimétrica: há uma cauda longa de municípios muito distantes da meta.")
print(f"⚠ Isso indica que políticas universais não serão suficientes — é necessário foco nos municípios da cauda.")

## 8. Features ML — Correlação entre Variáveis

In [ ]:
# Scatter: Taxa vs Gap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(features['taxa_alfabetizacao'], features['gap_meta_2030'], 
               alpha=0.3, s=10, c='#1565C0')
axes[0].set_xlabel('Taxa de Alfabetização Atual (%)')
axes[0].set_ylabel('Gap para Meta 2030')
axes[0].set_title('Taxa Atual vs Gap — Municípios', fontweight='bold')

# Scatter: Taxa vs Participação
axes[1].scatter(features['percentual_participacao'], features['taxa_alfabetizacao'], 
               alpha=0.3, s=10, c='#2E7D32')
axes[1].set_xlabel('Percentual de Participação (%)')
axes[1].set_ylabel('Taxa de Alfabetização (%)')
axes[1].set_title('Participação vs Taxa — Municípios', fontweight='bold')

plt.tight_layout()
plt.show()

# Insights ML
corr_gap = features[['taxa_alfabetizacao', 'gap_meta_2030', 'percentual_participacao', 'percentual_atingimento']].corr()
ja_atingiu_count = features['ja_atingiu_meta'].sum()
total_feat = len(features)
print('\n' + '='*60)
print('INSIGHTS — FEATURES PARA MACHINE LEARNING')
print('='*60)
print(f'Total de municípios no dataset de ML: {total_feat}')
print(f'Municípios que já atingiram a meta: {ja_atingiu_count} ({ja_atingiu_count/total_feat*100:.1f}%)')
print(f'Municípios que ainda não atingiram: {total_feat - ja_atingiu_count} ({(total_feat-ja_atingiu_count)/total_feat*100:.1f}%)')
print(f'\nCorrelações com taxa_alfabetizacao:')
print(f'  vs participação: {corr_gap.loc["taxa_alfabetizacao", "percentual_participacao"]:.3f}')
print(f'  vs gap_meta_2030: {corr_gap.loc["taxa_alfabetizacao", "gap_meta_2030"]:.3f} (correlação inversa esperada)')
print(f'\nPossíveis modelos:')
print(f'  1. Classificação: prever ja_atingiu_meta (0/1) → Logistic Regression, Random Forest')
print(f'  2. Regressão: prever taxa futura → Gradient Boosting, XGBoost')
print(f'  3. Clustering: agrupar municípios similares → K-Means, DBSCAN')
print(f'  4. Anomalias: detectar quedas abruptas → Isolation Forest')
print(f'\n✓ Dataset balanceado: {ja_atingiu_count/total_feat*100:.0f}% positivos vs {(total_feat-ja_atingiu_count)/total_feat*100:.0f}% negativos')

## 9. Resumo Executivo

### Principais Conclusões

**Panorama Nacional:**
1. A taxa nacional de alfabetização subiu de 55.9% (2023) para 66% (2025), um crescimento de +10.1 p.p. em 2 anos
2. A meta de 80% até 2030 exige um ritmo de ~2.8 p.p./ano — factível mas exige continuidade das políticas
3. Apenas o Ceará já atingiu a meta nacional (85.31%), servindo como referência de boas práticas

**Desigualdade Regional:**
4. Norte e Nordeste têm gap 2x maior que o Sul para a meta 2030
5. A diferença entre melhor (Sul) e pior região (Norte/Nordeste) chega a ~16 pontos percentuais
6. Mesmo dentro de uma região, há alta amplitude — indicando que soluções precisam ser localizadas, não apenas regionais

**Municípios em Risco:**
7. Existem municípios com taxa abaixo de 15% — precisariam crescer +11 p.p./ano, algo sem precedentes históricos
8. A maioria dos municípios críticos está no Nordeste (>70%), especialmente BA, PB, RN e SE
9. Municípios com baixa participação (<75%) tendem a ter dados menos confiáveis — possível viés de seleção

**Tendência e Projeção:**
10. Alguns estados regrediram (AM, RS) — queda na taxa indica problemas estruturais que precisam investigação
11. Estados com crescimento >5 p.p./ano (AL, AP, PI) demonstram que aceleração rápida é possível
12. Sem mudança de ritmo, ~30% dos estados NÃO atingirão a meta em 2030

**Streaming (AEEB 2025):**
13. Dados granulares por aluno permitem análises de equidade dentro do município (por escola, por rede)
14. A rede municipal concentra ~60% dos alunos avaliados e tem desempenho inferior à estadual na maioria dos estados
15. O ponto de corte de 743 na escala SAEB separa claramente dois grupos de alunos — a distribuição é bimodal em vários estados

**Implicações para Políticas Públicas:**
16. Políticas universais (iguais para todos) não resolverão o problema — é necessário focalização nos municípios críticos
17. O modelo de sucesso do Ceará (regime de colaboração estado-município + avaliação + incentivos) pode ser replicado
18. Monitoramento contínuo (streaming) permite intervenção precoce — identificar escolas em queda antes do fim do ano letivo

**Para Machine Learning:**
19. O dataset Gold suporta modelos preditivos: features históricas (taxa, metas, gap) + contextuais (região, rede, participação)
20. Caso de uso prioritário: sistema de alerta precoce que identifica municípios em risco de não atingir a meta intermediária (2026)

In [ ]:
# Resumo numérico
print('='*60)
print('RESUMO DO PIPELINE — CAMADA GOLD')
print('='*60)
print(f'\nTabelas analíticas geradas: 9')
print(f'Panorama: {len(panorama)} registros (Brasil + UF + Município)')
print(f'Municípios analisados: {len(criticos)}')
print(f'  - Críticos (gap >= 60): {len(criticos[criticos["nivel_criticidade"] == "Crítico"])}')
print(f'  - Muito Alto (gap >= 40): {len(criticos[criticos["nivel_criticidade"] == "Muito Alto"])}')
print(f'Alunos processados (streaming): {indicadores["total_alunos_avaliados"].sum():,.0f}')
print(f'Taxa nacional de alfabetização (streaming): {indicadores["total_alfabetizados"].sum() / indicadores["total_alunos_avaliados"].sum() * 100:.1f}%')
print('='*60)